In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os, h5py, math
import numpy as np, pandas as pd
import torch
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# --- 1) Load data dicts ---
file_path = "elucidata_ai_challenge_data.h5"
with h5py.File(file_path, "r") as f:
    train_images = {k: np.array(v) for k, v in f["images/Train"].items()}
    train_spots  = {k: np.array(v) for k, v in f["spots/Train"].items()}
    test_images  = {k: np.array(v) for k, v in f["images/Test"].items()}
    test_spots   = {k: np.array(v) for k, v in f["spots/Test"].items()}

# --- 2) Slide-specific pixel shifts (as before) ---
shifts = {
    "S_1":(0,0), "S_2":(0,0),
    "S_3":(0,0),   "S_4":(0,0),
    "S_5":(0,0),     "S_6":(0,0),
    # S_7: test → defaults to (0,0)
}

# --- 3) Hyperparams & Transforms ---
PATCH_SIZE     = 75
BATCH_SIZE     = 64
LR             = 0.0005
EPOCHS         = 80
NUM_CELL_TYPES = 35
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(0.1,0.1,0.1,0.05),
    transforms.RandomAffine(10, scale=(0.9,1.1)),
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tfm = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# --- 4) Dataset: returns (img5chan, coord2, label) or (img5chan, coord2, idx) ---
class HistologyDataset(Dataset):
    def __init__(self, images, spots, shifts, slides, transform, mode="Train"):
        self.mode      = mode
        self.transform = transform
        self.data      = []
        for slide in slides:
            img       = images[slide]
            spots_arr = spots[slide]
            dx,dy     = shifts.get(slide,(0,0))
            coords    = np.stack([spots_arr["x"]+dx, spots_arr["y"]+dy], axis=1)
            for i,(x,y) in enumerate(coords):
                lbl = spots_arr[i] if mode=="Train" else None
                self.data.append((slide,int(x),int(y),lbl))
        self.images = images

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        slide, x, y, label_row = self.data[idx]
        img = self.images[slide]
        h,w,_ = img.shape
        half = PATCH_SIZE//2

        # crop + pad patch
        y0,y1 = max(0,y-half), min(h,y+half)
        x0,x1 = max(0,x-half), min(w,x+half)
        patch = img[y0:y1, x0:x1]
        pad_y = (max(0,half-y),   max(0,(y+half)-h))
        pad_x = (max(0,half-x),   max(0,(x+half)-w))
        patch = np.pad(patch, (pad_y, pad_x, (0,0)), mode='constant', constant_values=0)
        img_t = self.transform(patch)  # 3×H×W

        # build 2-channel absolute coords
        x_norm = x / w
        y_norm = y / h
        H,W = img_t.shape[1:]
        x_map = torch.full((1,H,W), x_norm, dtype=torch.float32)
        y_map = torch.full((1,H,W), y_norm, dtype=torch.float32)
        img5  = torch.cat([img_t, x_map, y_map], dim=0)  # 5×H×W

        # build 2-dim normalized coord vector
        coord2 = torch.tensor([x_norm, y_norm], dtype=torch.float32)

        if self.mode=="Train":
            y = torch.tensor([label_row[f"C{j}"] for j in range(1,NUM_CELL_TYPES+1)],
                             dtype=torch.float32)
            return img5, coord2, y
        else:
            return img5, coord2, idx

# --- 5) Fourier MLP on 2-D coords ---
class FourierFeature(nn.Module):
    def __init__(self, in_dim=2, mapping_size=64, scale=10.0):
        super().__init__()
        B = torch.randn(mapping_size, in_dim) * scale
        self.register_buffer("B", B)

    def forward(self, x):
        x_proj = 2*math.pi * x @ self.B.t()        # [B,64]
        return torch.cat([x_proj.sin(), x_proj.cos()], dim=-1)  # [B,128]

# --- 6) Hybrid CoordConv + CoordMLP SpotNet ---
class SpotNet(nn.Module):
    def __init__(self, num_types=35, backbone="resnet34"):
        super().__init__()
        # --- Image tower with CoordConv(5→64) ---
        if backbone=="resnet34":
            cnn = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        else:
            cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        old1 = cnn.conv1
        new1 = nn.Conv2d(5, old1.out_channels,
                         kernel_size=old1.kernel_size,
                         stride=old1.stride,
                         padding=old1.padding,
                         bias=old1.bias is not None)
        with torch.no_grad():
            new1.weight[:,:3] = old1.weight  # copy RGB
            new1.weight[:,3:] = old1.weight[:,:2].mean(dim=1,keepdim=True)*0.0
        cnn.conv1 = new1
        dim = cnn.fc.in_features
        cnn.fc = nn.Identity()
        self.cnn = cnn

        # --- Coord MLP tower ---
        self.coord_ff  = FourierFeature(in_dim=2, mapping_size=64, scale=10.0)
        self.coord_mlp = nn.Sequential(
            nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU()
        )

        # --- Fusion head ---
        self.head = nn.Sequential(
            nn.Linear(dim + 128, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_types)
        )

    def forward(self, img5, coord2):
        f = self.cnn(img5)              # [B, dim]
        h = self.coord_ff(coord2)       # [B,128]
        h = self.coord_mlp(h)           # [B,128]
        x = torch.cat([f, h], dim=1)    # [B, dim+128]
        return self.head(x)             # [B,35]

# --- 7) Loss & validation ---
def spearman_loss(pred, target):
    rho=[]
    p_np=pred.detach().cpu().numpy()
    t_np=target.detach().cpu().numpy()
    for p,t in zip(p_np,t_np):
        r = spearmanr(p,t)[0]
        if not np.isnan(r): rho.append(r)
    return 1 - np.mean(rho)

def valid_loop(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for img5, coord2, y in loader:
            img5, coord2 = img5.to(DEVICE), coord2.to(DEVICE)
            preds.append(model(img5, coord2).cpu())
            trues.append(y)
    preds = torch.cat(preds); trues = torch.cat(trues)
    return spearman_loss(preds, trues)

In [ ]:
nb_type = "Train"

# --- 8) Training block ---
if nb_type=="Train":
    # slide split
    train_slides = [f"S_{i}" for i in range(1,6)]
    val_slides   = ["S_6"]

    ds_tr = HistologyDataset(train_images, train_spots, shifts,
                              train_slides, train_tfm, mode="Train")
    ds_va = HistologyDataset(train_images, train_spots, shifts,
                              val_slides,   eval_tfm,  mode="Train")
    loader_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
    loader_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model     = SpotNet(num_types=NUM_CELL_TYPES).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = nn.MSELoss()

    history, best_spear = {"train_loss":[], "val_spearman":[]}, -1
    print("🚀 Starting training...")
    for epoch in range(1, EPOCHS+1):
        # train epoch
        model.train()
        tloss=0
        for img5, coord2, y in loader_tr:
            img5, coord2, y = img5.to(DEVICE), coord2.to(DEVICE), y.to(DEVICE)
            p = model(img5, coord2)
            l = criterion(p,y)
            optimizer.zero_grad(); l.backward(); optimizer.step()
            tloss += l.item()
        tloss /= len(loader_tr)

        # validate
        vspear = 1 - valid_loop(model, loader_va)
        history["train_loss"].append(tloss)
        history["val_spearman"].append(vspear)
        print(f"[Epoch {epoch}] Train-Loss: {tloss:.4f}  Val-Spearman: {vspear:.4f}")
        if vspear > best_spear:
            best_spear = vspear
            torch.save(model.state_dict(), "best_model.pt")
            print(f"✅ Saved best (Spearman={best_spear:.4f})")

    print(f"\n✅ Training complete. Best Val Spearman = {best_spear:.4f}")

    # plot
    eps = list(range(1,EPOCHS+1))
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.plot(eps, history["train_loss"], label="Train Loss")
    plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.legend()
    plt.subplot(1,3,2)
    plt.plot(eps, history["val_spearman"], label="Val Spearman", color="C1")
    plt.xlabel("Epoch"); plt.ylabel("Spearman"); plt.legend()
    plt.subplot(1,3,3)
    plt.plot(eps, history["train_loss"]/history["val_spearman"], label="Train Loss")
    plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# # --- 9) Submission block (runs standalone) ---
# nb_type = "Submission"
# if nb_type=="Submission":
#     ds_ts = HistologyDataset(test_images, test_spots, shifts,
#                              ["S_7"], eval_tfm, mode="Submission")
#     loader_ts = DataLoader(ds_ts, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

#     model = SpotNet(num_types=NUM_CELL_TYPES).to(DEVICE)
#     model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE, weights_only=True))
#     model.eval()

#     subs=[]
#     with torch.no_grad():
#         for img5, coord2, idxs in loader_ts:
#             img5, coord2 = img5.to(DEVICE), coord2.to(DEVICE)
#             preds = model(img5, coord2).cpu().numpy()
#             for spot_idx, pred in zip(idxs.tolist(), preds):
#                 subs.append([spot_idx, *pred.tolist()])

#     cols = ["ID"] + [f"C{j}" for j in range(1,NUM_CELL_TYPES+1)]
#     sub_df = pd.DataFrame(subs, columns=cols)
#     sub_df.sort_values("ID", inplace=True)
#     sub_df.to_csv("submission.csv", index=False)
#     print("✅ submission.csv written:", sub_df.shape)

In [ ]:
# sub_df.info()